# C3：载重/风阻灵敏度分析
# C3: Vehicle Mass & Aerodynamic Drag Sensitivity Analysis

> **真实仿真声明**：本实验调用**真实 FASTSim 2.1.5 引擎**（NREL Rust 重写版），
> 11 次完整 UDDS 循环仿真，无 Python 替代模型。
> **Real-simulation note**: this lab drives the **real FASTSim 2.1.5 engine**
> (NREL Rust rewrite) for 11 full UDDS cycle simulations. No Python surrogate.

## 一、实验原理 | Principle

### 1.1 灵敏度系数是什么？| What is a sensitivity coefficient?

灵敏度系数回答："参数变化 1%，能耗变化百分之几？"——无量纲，可直接对比不同参数的重要性。

The sensitivity coefficient answers: "if a parameter changes by 1%, by what % does energy change?"
It is dimensionless, so mass and drag can be compared directly.

$$S_x = \frac{\Delta E / E}{\Delta x / x}$$

连续形式（本实验用对数回归估计）| Continuous form (estimated by log-log regression):

$$S_x = \frac{\partial E}{\partial x}\cdot\frac{x}{E} = \frac{\partial \ln E}{\partial \ln x}$$

### 1.2 能耗分解：∂E/∂m 与 ∂E/∂Cd 从哪来？
### Energy breakdown: where do ∂E/∂m and ∂E/∂Cd come from?

百公里电耗可分解为 | Energy per 100 km decomposes into:

$$E_{100km} = \frac{E_{roll} + E_{aero} + E_{inertial,net} + E_{aux}}{\eta_{sys}}$$

| 分量 Component | 公式 Formula | 对参数的依赖 Dependence |
|---|---|---|
| 滚阻 Rolling | $F_{rr} = \mu \cdot m \cdot g$ | $E_{roll} \propto m$ → $\partial E/\partial m \supset \mu g d/\eta > 0$ |
| 风阻 Aero drag | $F_{aero} = \tfrac{1}{2}\rho\, C_d\, A\, v^2$ | $E_{aero} \propto C_d$ → $\partial E/\partial C_d = \tfrac{1}{2}\rho A\int v^2 dt/\eta > 0$ |
| 惯性 Inertial | $\tfrac{1}{2}mv^2$ 每次加速 | 再生制动回收一部分：净耗 $\propto m(1-\eta_{regen})$ |

**关键推论 | Key insight**：
- $S_m$ 捕捉"滚阻 + 未被回收的惯性能"——市区频繁启停时很大
  ($S_m$ captures rolling + unrecovered inertial energy — large in stop-and-go city driving)
- $S_{Cd}$ 只捕捉风阻——随车速平方增长，高速时主导
  ($S_{Cd}$ captures aero drag only — grows with $v^2$, dominates at high speed)

### 1.3 为什么 UDDS 下质量更敏感？| Why is mass more sensitive on UDDS?

- **UDDS**：平均车速仅 31.5 km/h，全程 17 次停车 → 惯性能耗占比大 → $S_m > S_{Cd}$
  (UDDS: avg speed only 31.5 km/h with 17 stops → inertial losses dominate → $S_m > S_{Cd}$)
- **HWFET**（高速工况）：平均 77.6 km/h，$v^2$ 项放大 → $S_{Cd} > S_m$（见挑战任务）
  (HWFET highway: avg 77.6 km/h, the $v^2$ term wins → $S_{Cd} > S_m$, see challenge)
- 再生制动 $\eta_{regen}$ 越高，$S_m$ 越小——这是电动车与燃油车灵敏度排序不同的根本原因
  (Higher regen efficiency → smaller $S_m$ — the fundamental reason BEV sensitivity ordering differs from ICE vehicles)

### 1.4 FASTSim 2.1.5

NREL 开发的整车仿真工具，内置 70+ 实测车型参数。2.x 为 Rust 重写版，
本实验通过 `fastsim.vehicle.Vehicle` / `fastsim.simdrive.SimDrive` 直接调用。
NREL's vehicle simulator with 70+ measured vehicle models. v2.x is a Rust rewrite;
this lab calls it directly via `Vehicle` / `SimDrive` — no surrogate model.


## 二、实验目的 | Objectives

1. 掌握灵敏度系数的定义、公式与对数回归计算方法
   (Learn the definition, formula and log-log regression method of sensitivity coefficients)
2. 用**真实 FASTSim** 完成质量 ±20%、风阻系数 ±20% 的扫参仿真
   (Run real-FASTSim sweeps: mass ±20%, drag coefficient ±20%)
3. 量化 $S_m$、$S_{Cd}$，并解释"市区质量敏感、高速风阻敏感"的物理原因
   (Quantify $S_m$, $S_{Cd}$ and explain why mass dominates in the city but drag on the highway)


## 三、学生任务 | Your Tasks

1. **运行基线**：选 vnum=43（239 kW 大电机 BEV），跑 UDDS，记录基线能耗
   (Run baseline: vnum=43, a 239 kW BEV, on UDDS; record baseline energy)
2. **扫参**：质量取 80% / 90% / 100% / 110% / 120% 五点；风阻系数同样五点；各跑一次 UDDS
   (Sweep: 5 mass points and 5 drag points at 80/90/100/110/120%, one UDDS run each)
3. **计算**：对数回归求 $S_m$、$S_{Cd}$，画对比柱状图
   (Compute $S_m$, $S_{Cd}$ via log-log regression; bar-chart comparison)
4. **思考题**：
   - 为什么 UDDS 下 $S_m > S_{Cd}$？若关闭再生制动，排序会变吗？
     (Why is $S_m > S_{Cd}$ on UDDS? Would the ordering change with regen disabled?)
   - 若把扫描范围扩大到 ±40%，线性假设还成立吗？
     (If the sweep widened to ±40%, would the linearity assumption still hold?)


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')  # shared IEV CloudLab library
import numpy as np
import matplotlib.pyplot as plt
from fastsim.vehicle import Vehicle, DEFAULT_VEH_DB
from fastsim import cycle
from fastsim.simdrive import SimDrive
print('imports OK: real FASTSim 2.1.5')

VNUM = 43      # 239 kW BEV: motor strong enough that +/-20% mass stays linear
CYCLE = 'udds'

def run_energy(vnum, cycle_name, mass_scale=1.0, drag_scale=1.0):
    """Run one real FASTSim drive-cycle simulation with scaled mass/drag."""
    veh = Vehicle.from_file(DEFAULT_VEH_DB, vnum=vnum)  # real vehicle model
    base_mass = float(veh.veh_kg)
    base_cd = float(veh.drag_coef)
    # NOTE: set_veh_mass() takes no args; it sums component masses unless
    # veh_override_kg > 0, so we scale mass through the override field.
    veh.veh_override_kg = base_mass * mass_scale
    veh.set_veh_mass()
    veh.drag_coef = base_cd * drag_scale
    veh.set_derived()
    cyc = cycle.Cycle.from_file(cycle_name)  # real drive cycle
    sd = SimDrive(cyc, veh)
    sd.sim_drive()  # real physics simulation (Rust backend)
    kwh_per_mi = float(np.array(sd.electric_kwh_per_mi).mean())
    return kwh_per_mi * 62.137  # mi -> 100 km

# Baseline run
veh0 = Vehicle.from_file(DEFAULT_VEH_DB, vnum=VNUM)
print(f'vehicle: vnum={VNUM}, mass={float(veh0.veh_kg):.0f} kg, '
      f'Cd={float(veh0.drag_coef):.3f}, motor={float(veh0.mc_max_kw):.0f} kW')
base = run_energy(VNUM, CYCLE)
print(f'Baseline ({CYCLE}): {base:.2f} kWh/100km')
assert 8.0 <= base <= 12.0, f'baseline {base:.2f} out of expected range!'
print('baseline OK')


In [ ]:
# --- Mass sweep: 80% / 90% / 100% / 110% / 120% ---
scales = [0.8, 0.9, 1.0, 1.1, 1.2]
mass_E = [run_energy(VNUM, CYCLE, mass_scale=s) for s in scales]

# Physical sanity: heavier vehicle must consume more energy
assert all(b > a for a, b in zip(mass_E, mass_E[1:])), \
    'Energy must increase monotonically with mass!'
print('mass sweep (kWh/100km):', [f'{e:.2f}' for e in mass_E])

plt.figure(figsize=(7, 4))
plt.plot([s * 100 for s in scales], mass_E, 'o-', color='#007aff')
plt.xlabel('Mass (% of baseline)')
plt.ylabel('Energy (kWh/100km)')
plt.title('Mass sensitivity on UDDS (real FASTSim)')
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# --- Drag coefficient sweep: 80% / 90% / 100% / 110% / 120% ---
drag_E = [run_energy(VNUM, CYCLE, drag_scale=s) for s in scales]

# Physical sanity: larger drag must consume more energy
assert all(b > a for a, b in zip(drag_E, drag_E[1:])), \
    'Energy must increase monotonically with drag coefficient!'
print('drag sweep (kWh/100km):', [f'{e:.2f}' for e in drag_E])

plt.figure(figsize=(7, 4))
plt.plot([s * 100 for s in scales], drag_E, 's-', color='#34c759')
plt.xlabel('Drag coefficient (% of baseline)')
plt.ylabel('Energy (kWh/100km)')
plt.title('Aerodynamic drag sensitivity on UDDS (real FASTSim)')
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# --- Sensitivity coefficients via log-log linear regression ---
# S_x = d(ln E) / d(ln x): "% energy change per % parameter change"
log_s = np.log(scales)
S_m = np.polyfit(log_s, np.log(mass_E), 1)[0]
S_cd = np.polyfit(log_s, np.log(drag_E), 1)[0]
print(f'S_m  = {S_m:.3f}   (1% more mass -> {S_m:.2f}% more energy)')
print(f'S_Cd = {S_cd:.3f}  (1% more drag -> {S_cd:.2f}% more energy)')

# Assertions on real simulation results
assert S_m > 0 and S_cd > 0, 'Both sensitivities must be positive!'
# On UDDS (city, stop-and-go), inertial losses dominate -> mass wins.
# (On highway HWFET the ordering flips: see challenge task.)
assert S_m > S_cd, f'UDDS: expected S_m > S_Cd, got {S_m:.3f} vs {S_cd:.3f}'
print('All assertions passed: S_m > 0, S_Cd > 0, S_m > S_Cd on UDDS')

plt.figure(figsize=(6, 4))
plt.bar(['Mass $S_m$', 'Drag $S_{Cd}$'], [S_m, S_cd], color=['#007aff', '#34c759'])
plt.ylabel('Sensitivity coefficient')
plt.title('Sensitivity comparison on UDDS (real FASTSim)')
plt.grid(axis='y', alpha=0.3)
for i, v in enumerate([S_m, S_cd]):
    plt.text(i, v + 0.02, f'{v:.3f}', ha='center')
plt.show()


## 四、验证内容 | Verification Checklist

- [ ] 基线能耗在 8–12 kWh/100km 合理区间（实测约 9.4）
      (Baseline energy within 8–12 kWh/100km; measured ~9.4)
- [ ] 质量扫描 5 点严格单调递增（增重 → 能耗增加）
      (Mass sweep strictly increasing over the 5 points)
- [ ] 风阻扫描 5 点严格单调递增（增风阻 → 能耗增加）
      (Drag sweep strictly increasing over the 5 points)
- [ ] $S_m > 0$ 且 $S_{Cd} > 0$（灵敏度为正）
      ($S_m > 0$ and $S_{Cd} > 0$)
- [ ] UDDS 下 $S_m > S_{Cd}$（市区工况质量更敏感；实测约 0.61 vs 0.21）
      (On UDDS $S_m > S_{Cd}$; measured ~0.61 vs ~0.21)
- [ ] 全部 11 次仿真均调用真实 FASTSim，无 Python 替代
      (All 11 runs use the real FASTSim engine)


## 五、结果分析 | Results Analysis

### 5.1 灵敏度系数解读

UDDS 工况实测：
- **质量灵敏度 S_m = 0.610**：质量 +10% → 能耗 +6.1%
- **风阻灵敏度 S_Cd = 0.210**：风阻 +10% → 能耗 +2.1%
- Measured: S_m = 0.610, S_Cd = 0.210 under UDDS.

**物理意义**：S_m > S_Cd 说明在 UDDS 下，减重 1% 的节电效果约等于降风阻 3%。

### 5.2 为什么 UDDS 下质量更敏感？

UDDS 特征：平均车速仅 31.5 km/h，17 次启停/循环。
- 低速 → 空气阻力小
- 频繁加减速 → 惯性能耗主导
- UDDS: low speed → aero drag small; frequent accel → inertial energy dominates.

### 5.3 HWFET 反转：高速下风阻主导

实测 HWFET（高速工况）：S_m = 0.418，S_Cd = 0.487，**S_Cd > S_m**，与 UDDS 相反！

**教学价值**：没有"绝对"的灵敏度排序，必须结合工况看。这就是为什么电动车高速续航"打折"更严重。
No absolute sensitivity ranking — it depends on the drive cycle.

### 5.4 工程应用

| 场景 | 优先优化 |
|---|---|
| 城市通勤车 | 轻量化（质量敏感） |
| 高速长途车 | 低风阻造型（CdA 敏感） |

**设计启示**：灵敏度分析是"钱花在哪最值"的定量依据。


## 六、常见报错说明 | Troubleshooting

| 报错信息 Error | 中文说明 | 解决方法 Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'fastsim'` | 内核选错了，FASTSim 只装在 py310-fastsim 环境 | 右下角 Kernel → 切换为 **Python 3.10 (FASTSim)** 后重启 |
| `TypeError: set_veh_mass() takes 1 positional argument` | `set_veh_mass()` 不接受传参，它按部件求和 | 用 `veh.veh_override_kg = ...` 再调 `set_veh_mass()`（见本实验代码） |
| 质量增加但能耗反而下降 | 小电机车型（如 vnum=16，50 kW）+ 大幅增重会触发 FASTSim 功率限制，模型进入非物理区间 | 换大电机车型（如 vnum=43，239 kW）或把扫描范围缩小到 ±10% |
| 图像中文显示为方块 □□□ | 镜像缺少中文字体 | 绘图标签一律用**英文**（本实验已遵守） |
| `assert S_m > S_cd` 失败 | 可能误用了 HWFET 数据或车辆模型被改动 | 确认 `CYCLE='udds'` 且未改动基线车辆参数 |


## 🏆 挑战任务 | Challenge

1. **高速反转验证**：把 `CYCLE` 改为 `'hwfet'` 重跑全实验，验证 $S_{Cd} > S_m$
  （实测 HWFET：$S_m \approx 0.42$，$S_{Cd} \approx 0.49$）。用 $F_{aero} \propto v^2$ 解释。
   (Switch `CYCLE` to `'hwfet'` and re-run: verify the crossover $S_{Cd} > S_m$.
   Measured: $S_m \approx 0.42$, $S_{Cd} \approx 0.49$. Explain with $F_{aero} \propto v^2$.)
2. **再生制动的影响**：查 `Vehicle.regen_a / regen_b` 参数，尝试削弱再生制动，观察 $S_m$ 如何变化。
   (Weaken regen via `Vehicle.regen_a/regen_b` and watch how $S_m$ responds.)
3. **串联到能量管理链**：把本实验的 $S_m$、$S_{Cd}$ 存入 `labs/data/results/C3_sensitivity.json`
   （`from common.vehicle_library import save_result`），供 E2 优化时作为权重参考。
   (Save your $S_m$/$S_{Cd}$ into the shared results store for the E2 optimizer chain.)


---

## 下一个实验 | Next →

**[C4_ems_comparison](./C4_ems_comparison.ipynb)**

> **所需内核**：`py310-fastsim` | **Required kernel**: `py310-fastsim`
